# OpenPlaque — Plaque + Inflammation Final Visualization v2.2

Single **Runtime → Run all** notebook with explicit cache/recompute controls.

This notebook reproduces the accepted plaque and direct-PCAT endpoint, generates the corrected plaque and inflammation charts, and exports the full result set.

The expensive value stages are cache-aware:

- plaque/inflammation endpoint tables can be reused or forced to regenerate;
- validated source-space RCA/LAD/LCX PCAT voxel arrays can be reused or forced to reconstruct;
- cached voxel arrays are **always revalidated** against the locked mean HU and fat-voxel counts before reuse;
- if a requested cache is missing, unreadable, or invalid, the notebook automatically recomputes it.

Figures are **always regenerated** from the selected numerical values so stale figures are never silently reused.

Research boundaries remain unchanged: OpenPlaque plaque values are research best-estimate proxies, not Cleerly outputs. Direct PCAT is not proprietary Caristo FAI-Score. LM inflammation is not standardized. C6 remains an LCX-like structural research segment.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =========================
# USER CONTROLS
# =========================

from pathlib import Path

# DRIVE_ROOT is the parent Google Drive folder that contains all established
# OpenPlaque caches and prior research outputs used by this notebook.
DRIVE_ROOT = Path("/content/drive/MyDrive/OpenPlaque")

# OUT is the folder where this notebook stores its tables, validated voxel
# caches, figures, HTML report, and optional ZIP. Re-running with the same OUT
# allows the cache Booleans below to reuse validated results.
OUT = DRIVE_ROOT / "Plaque_Inflammation_Final_Visualization_v2_2"

# If True, reuse the consolidated plaque/inflammation endpoint tables already
# present in OUT/endpoint when they are complete and readable. If the cache is
# unavailable or invalid, it is regenerated automatically. If False, always
# regenerate the endpoint tables from the established upstream Drive caches.
USE_CACHED_ENDPOINT_VALUES = True

# If True, reuse the cached RCA/LAD/LCX source-space PCAT voxel arrays in OUT
# when available. They are always revalidated against the locked mean HU and
# fat-voxel counts before reuse. Missing, corrupt, or invalid caches are
# automatically recomputed. If False, force source-space voxel reconstruction.
USE_CACHED_PCAT_VOXELS = True

# If True, display every final PNG figure inline in the Colab notebook.
# If False, figures are still regenerated and saved to Drive but not displayed.
DISPLAY_FIGURES = True

# If True, package the entire output folder (tables, arrays, figures, JSON,
# HTML report, and endpoint subfolder) into one ZIP. If False, skip ZIP export.
EXPORT_ZIP = True

OUT.mkdir(parents=True, exist_ok=True)

print("Output folder:", OUT)
print("USE_CACHED_ENDPOINT_VALUES =", USE_CACHED_ENDPOINT_VALUES)
print("USE_CACHED_PCAT_VOXELS =", USE_CACHED_PCAT_VOXELS)
print("DISPLAY_FIGURES =", DISPLAY_FIGURES)
print("EXPORT_ZIP =", EXPORT_ZIP)

In [ ]:
# Required upstream inputs. These are established source-space or locked
# research caches; they are not the optional v2.2 caches controlled above.

REQUIRED = [
    DRIVE_ROOT / "UCLA_Plaque_Type_Estimates" / "best_estimate_plaque_types_by_artery.csv",
    DRIVE_ROOT / "Cache" / "Secondary_3D_Vesselness_Topology_v1" / "series7_int16.npy",
    DRIVE_ROOT / "Cache" / "Secondary_3D_Vesselness_Topology_v1" / "series7_int16.json",
    DRIVE_ROOT / "RCA_Ostium_TotalSegmentator" / "aorta_series7_totalseg.nii.gz",

    DRIVE_ROOT / "PCAT_RCA_10_50" / "rca_centerline_smoothed_zyx.csv",
    DRIVE_ROOT / "PCAT_RCA_10_50" / "pcat_local_radius_profile.csv",
    DRIVE_ROOT / "RCA_Plaque_PCAT_Research_Lock_v1" / "summary.json",
    DRIVE_ROOT / "RCA_Plaque_PCAT_Research_Lock_v1" / "RCA_locked_research_plaque_PCAT_profile_10_50.csv",

    DRIVE_ROOT / "LAD_Source_Space_PCAT_Feasibility_v1" / "LAD_PCAT_source_geometry.csv",
    DRIVE_ROOT / "LAD_Source_Space_PCAT_Feasibility_v1" / "summary.json",
    DRIVE_ROOT / "LAD_Source_Space_PCAT_Feasibility_v1" / "frozen_LAD_PCAT_longitudinal.csv",

    DRIVE_ROOT / "LCX_Structural_Source_QC_Freeze_v1" / "LCX_structural_dense_source_QC.csv",
    DRIVE_ROOT / "LCX_OM_Source_Space_Composition_PCAT_Feasibility_v1" / "summary.json",
    DRIVE_ROOT / "LCX_OM_Source_Space_Composition_PCAT_Feasibility_v1" / "C6_PCAT_longitudinal_primary.csv",
    DRIVE_ROOT / "LCX_OM_Source_Space_Composition_PCAT_Feasibility_v1" / "C7_PCAT_longitudinal_primary.csv",
]

missing = [str(p) for p in REQUIRED if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing required upstream inputs:\n" + "\n".join(missing))

print("All required upstream inputs found.")

In [ ]:
import shutil, sys, subprocess, json
from pathlib import Path

# OPENPLAQUE_PIN is the exact source/tests/docs commit used for this notebook,
# making the run reproducible even if the branch changes later.
OPENPLAQUE_PIN = "97e8b0c4634bf5a03b50347a47531ffac79e025e"

# OPENPLAQUE_BRANCH is the GitHub branch containing the plaque/inflammation
# endpoint and final visualization code.
OPENPLAQUE_BRANCH = "plaque-inflammation-best-estimates-from-main"

if Path("/content/OpenPlaque").exists():
    shutil.rmtree("/content/OpenPlaque")

!git clone -q --branch {OPENPLAQUE_BRANCH} https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!git -C /content/OpenPlaque checkout -q {OPENPLAQUE_PIN}

%pip install -q /content/OpenPlaque

for name in list(sys.modules):
    if name == "openplaque" or name.startswith("openplaque."):
        del sys.modules[name]

actual = subprocess.check_output(
    ["git","-C","/content/OpenPlaque","rev-parse","HEAD"], text=True
).strip()
print("OpenPlaque pin:", actual)
assert actual == OPENPLAQUE_PIN

In [ ]:
# Run synthetic/unit tests before using study data.
from openplaque.plaque_inflammation_final_visualization_v2 import synthetic_self_test

print(synthetic_self_test())
!cd /content/OpenPlaque && pytest -q   tests/test_plaque_inflammation_best_estimates_v1.py   tests/test_plaque_inflammation_final_visualization_v2.py

In [ ]:
# Run the complete endpoint + corrected visualization pipeline using the
# user-selected cache/recompute settings.
from openplaque.plaque_inflammation_final_visualization_v2 import run

summary = run(
    drive_root=str(DRIVE_ROOT),
    output_dir=str(OUT),
    use_cached_endpoint_values=USE_CACHED_ENDPOINT_VALUES,
    use_cached_pcat_voxels=USE_CACHED_PCAT_VOXELS,
    export_zip=EXPORT_ZIP,
)

print(json.dumps(summary, indent=2))

print("\nCache usage in this run:")
print("  Endpoint tables reused:", summary["cache_usage"]["endpoint_values_reused"])
print("  PCAT voxel arrays reused:", summary["cache_usage"]["pcat_voxels_reused"])

In [ ]:
# Review the exact endpoint values and the voxel-cache validation.
import pandas as pd
from IPython.display import display

endpoint = OUT / "endpoint"

plaque = pd.read_csv(endpoint / "plaque_best_estimates_by_vessel.csv")
aggregate = pd.read_csv(endpoint / "major_vessel_aggregate.csv")
inflammation = pd.read_csv(endpoint / "inflammation_best_estimates_by_vessel.csv")
validation = pd.read_csv(OUT / "pcat_voxel_reconstruction_validation.csv")
bands = pd.read_csv(OUT / "pcat_voxel_hu_band_decomposition.csv")

print("Plaque best estimates")
display(plaque[[
    "vessel",
    "tpv_best_estimate_mm3",
    "tpv_strict_or_known_lower_mm3",
    "tpv_candidate_envelope_upper_mm3",
    "ncpv_best_estimate_mm3",
    "lap_best_estimate_mm3",
    "calcified_plaque_volume_mm3",
    "confirm2_tpv_stage",
    "absolute_volume_confidence",
]])

print("\nMajor-vessel aggregate")
display(aggregate)

print("\nLocked direct PCAT endpoint")
display(inflammation[[
    "vessel",
    "pcat_mean_hu_best_estimate",
    "pcat_segment_length_mm",
    "segment_coverage_fraction",
    "caristo_comparability",
    "fai_score",
    "confidence",
]])

print("\nPCAT voxel-cache/source reconstruction validation")
display(validation)

if not validation["voxel_distribution_validation_pass"].all():
    raise RuntimeError("PCAT voxel values do not reproduce the locked endpoints.")

print("\nTrue source-space PCAT voxel attenuation bands")
display(bands)

In [ ]:
# FIGURES lists the corrected PNG files generated on every run. They are
# regenerated from the selected cached or recomputed numerical values.
FIGURES = [
    "01_plaque_composition_corrected.png",
    "02_inflammation_mean_pcat_corrected.png",
    "03_inflammation_voxel_hu_decomposition.png",
    "04_inflammation_voxel_hu_histograms.png",
    "05_inflammation_longitudinal_profiles.png",
    "06_inflammation_longitudinal_heatmap.png",
    "07_rca_radial_pcat_gradient_descriptive.png",
]

if DISPLAY_FIGURES:
    from IPython.display import Image, display

    for name in FIGURES:
        p = OUT / name
        if not p.is_file():
            raise FileNotFoundError(p)
        print("\n" + name)
        display(Image(filename=str(p), width=1200))
else:
    print("Inline figure display disabled. All PNGs were still written to:", OUT)

In [ ]:
# Display the complete HTML report regardless of the inline-PNG setting.
from IPython.display import HTML, display

report = OUT / "OPENPLAQUE_PLAQUE_INFLAMMATION_FINAL_VISUALIZATION_V2_REPORT.html"
display(HTML(report.read_text()))

In [ ]:
# Verify all required deliverables and, when requested, the ZIP archive.
ZIP_PATH = OUT / "OPENPLAQUE_PLAQUE_INFLAMMATION_FINAL_VISUALIZATION_V2_RESULTS.zip"

expected = [
    "run_state.json",
    "summary.json",
    "pcat_voxel_reconstruction_validation.csv",
    "pcat_voxel_hu_band_decomposition.csv",
    "pcat_voxel_values.npz",
    "pcat_voxel_geometry_extras.npz",
    "pcat_longitudinal_profiles_used.csv",
    "rca_locked_radial_profile_used.csv",
    "01_plaque_composition_corrected.png",
    "02_inflammation_mean_pcat_corrected.png",
    "03_inflammation_voxel_hu_decomposition.png",
    "04_inflammation_voxel_hu_histograms.png",
    "05_inflammation_longitudinal_profiles.png",
    "06_inflammation_longitudinal_heatmap.png",
    "07_rca_radial_pcat_gradient_descriptive.png",
    "OPENPLAQUE_PLAQUE_INFLAMMATION_FINAL_VISUALIZATION_V2_REPORT.html",
]

if EXPORT_ZIP:
    expected.append("OPENPLAQUE_PLAQUE_INFLAMMATION_FINAL_VISUALIZATION_V2_RESULTS.zip")

missing = [x for x in expected if not (OUT / x).exists()]
if missing:
    raise RuntimeError("Missing outputs: " + str(missing))

state = json.loads((OUT / "run_state.json").read_text())
if state.get("status") != "COMPLETE":
    raise RuntimeError("Run state not COMPLETE: " + str(state))

print("COMPLETE")
print("Endpoint cache reused:", state.get("endpoint_values_reused"))
print("PCAT voxel cache reused:", state.get("pcat_voxels_reused"))

if EXPORT_ZIP:
    print("ZIP:", ZIP_PATH)
    print("ZIP size (MB):", ZIP_PATH.stat().st_size / 1e6)
else:
    print("ZIP export was disabled by EXPORT_ZIP = False")

print("\nTop-level output files:")
for p in sorted(OUT.iterdir()):
    if p.is_file():
        print(" ", p.name)